In [ ]:
import numpy
import torch.nn as nn
import torch

#input parameters
n=336 #batch size
l=50 #sequence len
c=4 #embedding dimension

#model parameters
k = 3 #kernel size
s = 4 #stride
p = 2 #pool size

#model parameters
h1 = 32 #first lstm hidden size
h2  = 16 #second lstm hidden size

m = nn.AvgPool1d(p)


In [2]:
#Generate input [336, 50, 4]
rand_arr = torch.rand(n,l,c) 
print(rand_arr.shape)

torch.Size([336, 50, 4])


In [3]:
#Permute into correct shape (batch, embeddings, sequence len)
rand_arr_permute = rand_arr.permute(0,2,1)
print(rand_arr_permute.shape)

torch.Size([336, 4, 50])


In [4]:
conv1 = nn.Conv1d(out_channels=32, in_channels=c, kernel_size=k)
out1 = conv1(rand_arr_permute)
print(out1.shape)
out1= m(out1)
print(out1.shape)

torch.Size([336, 32, 48])
torch.Size([336, 32, 24])


In [5]:
out_m = out1.permute(0,2,1)
print(out_m.shape)

torch.Size([336, 24, 32])


In [6]:

lstm2 = nn.LSTM(input_size=h1, hidden_size=h2, num_layers=1, bidirectional=False, batch_first=True)
out2, _ = lstm2(out_m)
print(out2.shape)

torch.Size([336, 24, 16])


In [8]:
lstm4 = nn.LSTM(input_size=h2, hidden_size=h1, num_layers=1, bidirectional=False, batch_first=True)
out3, _ = lstm4(out2)
print(out3.shape)

torch.Size([336, 24, 32])


In [10]:
out_n = out3.permute(0,2,1)
print(out_n.shape)

torch.Size([336, 32, 24])


In [11]:
conv4 = nn.ConvTranspose1d(out_channels=c, in_channels=32, kernel_size=k, stride=s)
out4 = conv4(out_n)
print(out4.shape)
out4= m(out4)
print(out4.shape)

torch.Size([336, 4, 95])
torch.Size([336, 4, 47])


In [12]:
dense = nn.Linear(out4.shape[-1], l)
out5 = dense(out4)
print(out5.shape)


torch.Size([336, 4, 50])


In [ ]:
class ConvLSTMAutoEncoder(nn.Module):
    def __init__(self):
        super(ConvLSTMAutoEncoder, self).__init__()
        #from input
        self.features = 5
        self.seq_len = 50
        #for cnn
        self.kernel_size = 7
        self.pool_size = 2
        #for lstm
        self.hidden_1 = 32
        self.hidden_2 = 16

        self.encode1 = nn.Sequential(
            nn.Conv1d(out_channels=32, in_channels=self.features, kernel_size=self.kernel_size),
            nn.Tanh(),
            nn.AvgPool1d(self.pool_size),
            nn.Dropout(0.2)
        )

        self.encode2 = nn.LSTM(input_size=self.hidden_1, hidden_size=self.hidden_2, num_layers=1, bidirectional=False, batch_first=True)

        self.decode1 = nn.LSTM(input_size=self.hidden_2, hidden_size=self.hidden_1, num_layers=1, bidirectional=False, batch_first=True)

        self.decode2 = nn.Sequential(
            nn.ConvTranspose1d(out_channels=self.seq_len, in_channels=32, kernel_size=self.kernel_size, stride=4),
            nn.AvgPool1d(self.pool_size),
            nn.Dropout(0.2)
        )
        self.dense = nn.Linear(47, self.seq_len)

    def forward(self, x):
        #cnn takes input of shape (batch_size, channels, seq_len)
        x = x.permute(0,2,1)
        x = self.encode1(x)
        # lstm takes input of shape (batch_size, seq_len, input_size)
        x = x.permute(0,2,1)
        x, _ = self.encode2(x)
        x, _ = self.decode1(x)
        #cnn takes input of shape (batch_size, channels, seq_len)
        x = x.permute(0,2,1)
        x = self.decode2(x)
        x = self.dense(x)
        #input was of shape (batch_size, seq_len, features)
        x = x.permute(0,2,1)
        return x